In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from skimage import io  #read images
from skimage.filters import roberts, sobel, scharr, prewitt, gabor_kernel

import cv2
import shutil
from google.colab import files

from google.colab.patches import cv2_imshow
import os

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import precision_recall_curve, roc_curve, auc
from sklearn.metrics import classification_report
from sklearn.model_selection import learning_curve, GridSearchCV

from sklearn.decomposition import PCA

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
trainingData = pd.read_csv("./data/trainset.csv") #, index_col=0)
testingData = pd.read_csv("./data/testset.csv") #, index_col=0)

In [ ]:
counts = trainingData["Class"].value_counts()
sorted_counts = counts.sort_index()
trainingData["Class"].value_counts()

In [ ]:
plt.pie(sorted_counts.tolist(), labels = sorted_counts.index, autopct='%.0f%%')
plt.title("Target Distribution")
plt.show()

In [ ]:
trainingData.head()

In [ ]:
def create_gaborFilter():
  filters = []
  for theta in range(2):
    theta = theta / 4.0 * np.pi
    for sigma in (3, 5):
      for lamda in np.arange(0, np.pi, np.pi/4):
        for gamma in (0.05, 0.5):
          #gaborLabel = 'Gabor' + str(count)
          kernel = cv2.getGaborKernel((5, 5), sigma, theta, lamda, gamma, 0, ktype = cv2.CV_32F)
          filters.append(kernel)
          #filtered_IMG = cv2.filter2D(img, cv2.CV_8UC3, kernel)
          #filtered_img = filtered_IMG.reshape(-1)
          #df[gaborLabel] = filtered_img
  return filters

In [ ]:
def apply_filter(img, filters):
  # This general function is designed to apply filters to our image
  # First create a numpy array the same size as our input image

  newimage = np.zeros_like(img)
  # Starting with a blank image, we loop through the images and apply our Gabor Filter
  # On each iteration, we take the highest value (super impose), until we have the max value across all filters
  # The final image is returned
  depth = -1 # remain depth same as original image

  for kern in filters:  # Loop through the kernels in our GaborFilter
    image_filter = cv2.filter2D(img, depth, kern)  #Apply filter to image

    # Using Numpy.maximum to compare our filter and cumulative image, taking the higher value (max)
    np.maximum(newimage, image_filter, newimage)   #(arr1, arr2, result)
  return newimage

In [ ]:
gfilters = create_gaborFilter()
image_g = apply_filter(resized_img, gfilters)

cv2_imshow(resized_img)
cv2_imshow(image_g)

In [ ]:
image_g.shape

In [ ]:
#Feature Extraction:
#Square Box Filter

training_ids = set(trainingData['Id'])
testing_ids = set(testingData['Id'])

for image in os.listdir("./data/ALLImages"):

  image_id = int(image.split('.')[0])
  path = "./data/ALLImages/" + image
  img = cv2.imread(path)
  resized_img = cv2.resize(img, (200, 200))


  # Apply the Scharr filter in x and y directions
  scharr_x = cv2.Scharr(resized_img, cv2.CV_16S, 1, 0)
  scharr_y = cv2.Scharr(resized_img, cv2.CV_16S, 0, 1)

  # Convert the filtered images back to CV_8U type
  scharr_x_abs = cv2.convertScaleAbs(scharr_x)
  scharr_y_abs = cv2.convertScaleAbs(scharr_y)

  # Combine the x and y filtered images to get the final edge image
  Scharr_filtered_img = cv2.addWeighted(scharr_x_abs, 0.5, scharr_y_abs, 0.5, 0)

  #

  gray_img = cv2.cvtColor(Scharr_filtered_img, cv2.COLOR_BGR2GRAY)
  img2 = gray_img.reshape(1, -1)

  if image_id in training_ids:
    for train_id in training_ids:
      if image_id == train_id:
        row_index = trainingData.index[trainingData['Id'] == image_id][0]
        # trainingData.at[row_index, 'IDFeature'] = image_id
        # Save the pixel values as separate features
        for idx, pixel_value in enumerate(img2.flatten()):
          feature_name = f"img_feature{idx}"
          trainingData.at[row_index, feature_name] = pixel_value
        break

  if image_id in testing_ids:
    for test_id in testing_ids:
      if image_id == test_id:
        row_index = testingData.index[testingData['Id'] == image_id][0]
        #testingData.at[row_index, 'IDFeature'] = image_id

        for idx, pixel_value in enumerate(img2.flatten()):
          feature_name = f"img_feature{idx}"
          testingData.at[row_index, feature_name] = pixel_value
        break

trainingData.to_csv("./data/updated_training_scharrFilter_200X200.csv", index = False)
testingData.to_csv("./data/updated_testing_scharrFilter_200X200.csv", index = False)

In [ ]:
#Feature Extraction:
#bilateral Filter

training_ids = set(trainingData['Id'])
testing_ids = set(testingData['Id'])

for image in os.listdir("./data/ALLImages"):

  image_id = int(image.split('.')[0])
  path = "./data/ALLImages/" + image
  img = cv2.imread(path)
  resized_img = cv2.resize(img, (200, 200))

  #Applying The Filter:
  #
  # Apply bilateral filter
  d = 9         # Diameter of pixel neighborhood
  sigmaColor = 75  # Filter sigma in the color space
  sigmaSpace = 75  # Filter sigma in the coordinate space
  bilateralFilter_filtered_img = cv2.bilateralFilter(resized_img, d, sigmaColor, sigmaSpace)
  #

  gray_img = cv2.cvtColor(bilateralFilter_filtered_img, cv2.COLOR_BGR2GRAY)
  img2 = gray_img.reshape(1, -1)

  if image_id in training_ids:
    for train_id in training_ids:
      if image_id == train_id:
        row_index = trainingData.index[trainingData['Id'] == image_id][0]
        # trainingData.at[row_index, 'IDFeature'] = image_id

        # Save the pixel values as separate features
        for idx, pixel_value in enumerate(img2.flatten()):
          feature_name = f"img_feature{idx}"
          trainingData.at[row_index, feature_name] = pixel_value
        break

  if image_id in testing_ids:
    for test_id in testing_ids:
      if image_id == test_id:
        row_index = testingData.index[testingData['Id'] == image_id][0]
        #testingData.at[row_index, 'IDFeature'] = image_id

        for idx, pixel_value in enumerate(img2.flatten()):
          feature_name = f"img_feature{idx}"
          testingData.at[row_index, feature_name] = pixel_value
        break

trainingData.to_csv("./data/updated_training_bilateralFilter_200X200.csv", index = False)
testingData.to_csv("./data/updated_testing_bilateralFilter_200X200.csv", index = False)

In [ ]:
#Feature Extraction:
#Scharr Filter:

training_ids = set(trainingData['Id'])
testing_ids = set(testingData['Id'])

for image in os.listdir("./data/ALLImages"):

  image_id = int(image.split('.')[0])
  path = "./data/ALLImages/" + image
  img = cv2.imread(path)
  resized_img = cv2.resize(img, (200, 200))

  #Applying The Filter:
  #


  #

  if image_id in training_ids:
    for train_id in training_ids:
      if image_id == train_id:
        row_index = trainingData.index[trainingData['Id'] == image_id][0]
        # trainingData.at[row_index, 'IDFeature'] = image_id

        # Save the pixel values as separate features
        for idx, pixel_value in enumerate(img2.flatten()):
          feature_name = f"img_feature{idx}"
          trainingData.at[row_index, feature_name] = pixel_value
        break

  if image_id in testing_ids:
    for test_id in testing_ids:
      if image_id == test_id:
        row_index = testingData.index[testingData['Id'] == image_id][0]
        #testingData.at[row_index, 'IDFeature'] = image_id

        for idx, pixel_value in enumerate(img2.flatten()):
          feature_name = f"img_feature{idx}"
          testingData.at[row_index, feature_name] = pixel_value
        break

trainingData.to_csv("./data/updated_training_Scharr Filter_200X200.csv", index = False)
testingData.to_csv("./data/updated_testing_Scharr Filter_200X200.csv", index = False)

In [ ]:
#Saving Normal Images in NormalImagesFolder & BenignImages:
training_ids = set(trainingData['Id'])
testing_ids = set(testingData['Id'])

for image in os.listdir("images"):
  image_id = int(image.split('.')[0])
  path = "images/" + image
  img = cv2.imread(path)
  gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  if image_id in training_ids:
    for train_id in training_ids:
      if image_id == train_id:
        row_index = trainingData.index[trainingData['Id'] == image_id][0]
        class_label = trainingData.loc[row_index, 'Class']
        if class_label == "Malignant":
          image_filename = os.path.basename(path)
          target_image_path = os.path.join("MalignantImages", image_filename)
          shutil.copy2(path, target_image_path)
        break

In [ ]:
# #Download Folders:

# # Folder to be downloaded
# folder_to_download = "MalignantImages"

# # ZIP archive name
# zip_filename = "MalignantImages.zip"

# # Create a ZIP archive of the folder
# shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', folder_to_download)

# # Download the ZIP archive
# files.download(zip_filename)

In [ ]:
trainingData.head()

In [ ]:
trainingDataOriginal = pd.read_csv("updated_trainingOriginalP.csv")
trainingDataOriginal.head()

In [ ]:
trainingData = pd.read_csv("./data/updated_trainingOriginalP100X100.csv")
testingData = pd.read_csv("./data/updated_testingOriginalP100X100.csv")

In [ ]:
# trainingData = pd.read_csv("./data/updatedTrainingDataUsingReflection.csv")
# testingData = pd.read_csv("./data/updated_testingOriginalP_200X200.csv")

In [ ]:
trainingData.head()

In [ ]:
x = trainingData.drop(columns = ['Class', 'Id', 'Unnamed: 0'])
y = trainingData['Class']

X_train, X_val, y_train, y_val = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
testData = testingData.drop(columns = ['Id', 'Unnamed: 0'])

In [ ]:
pca = PCA(n_components = 80)

X_train = pca.fit_transform(X_train)
X_val = pca.transform(X_val)
testData = pca.transform(testData)

In [ ]:
x.shape

In [ ]:
testData.shape

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]}

#Random Forest
rf = RandomForestClassifier(random_state = 42, criterion = "entropy")

stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform grid search with cross-validation
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=stratified_cv)
grid_search.fit(X_train, y_train)

# Print the best parameters and best score from the grid search
print("Best Parameters:", grid_search.best_params_)

# Evaluate the best model on the test set
best_rf_model = grid_search.best_estimator_
predictions = best_rf_model.predict(X_val)

print(classification_report(y_val, predictions))

In [ ]:
#Random Forest
rf = RandomForestClassifier()

rf.fit(X_train, y_train)
predictions = rf.predict(X_val)
print(classification_report(y_val, predictions))

In [ ]:
#KNeighborsClassifier:
KNeighbors = KNeighborsClassifier(n_neighbors=3)
KNeighbors.fit(X_train, y_train)
predictions = KNeighbors.predict(X_val)

print(classification_report(y_val, predictions))

In [ ]:
#Gradient Boosting Classifier:

GBClassifier = GradientBoostingClassifier()

GBClassifier.fit(X_train, y_train)
predictions = GBClassifier.predict(X_val)

print(classification_report(y_val, predictions))

In [ ]:
HistModel = HistGradientBoostingClassifier(random_state= 42)

param_grid = {
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_bins': [255, 512]
    }

# Create a StratifiedKFold cross-validator
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform grid search with StratifiedKFold cross-validation
grid_search = GridSearchCV(estimator=HistModel, param_grid=param_grid, cv=stratified_cv)
grid_search.fit(X_train, y_train)

# Print the best parameters and best score from the grid search
print("Best Parameters:", grid_search.best_params_)

# Evaluate the best model on the test set
best_hist_gb_model = grid_search.best_estimator_
predictions = best_hist_gb_model.predict(X_val)

print(classification_report(y_val, predictions))

In [ ]:
HistModel = HistGradientBoostingClassifier()

HistModel.fit(X_train, y_train)
predictions = HistModel.predict(X_val)

print(classification_report(y_val, predictions))

In [ ]:
from sklearn.metrics import f1_score

# Compute F1 scores for training and validation sets
train_f1_scores = [f1_score(y_train, y_pred, average='weighted') for y_pred in HistModel.staged_predict(X_train)]
validation_f1_scores = [f1_score(y_val, y_pred, average='weighted') for y_pred in HistModel.staged_predict(X_val)]


fig, ax = plt.subplots(figsize = (10, 5))
ax.plot(train_f1_scores, label = "Train")
ax.plot(validation_f1_scores, "--", label = "Validation")
ax.set_xlabel("Boosting Iteration (Trees)")
ax.set_ylabel("f1 score")
ax.legend()

plt.show()

In [ ]:
#test = pd.read_csv("updated_testingOriginalP.csv")
testData = testingData.drop(columns = ['Unnamed: 0', 'Id'])
#predictedV = rf.predict(testData)

In [ ]:
testData.shape

In [ ]:
predictedV = HistModel.predict(testData)

In [ ]:
IDDF = testingData['Id']

predictionsDF = pd.DataFrame({'Id': IDDF, 'Class': predictedV})

predictionsDF.to_csv("submissionH.csv", index=False)